# RAIN RainNet v2 Training (46-param)

Train the RainNet v2 transformer to predict 46 DSP parameters from mel spectrograms.

**Requirements:** GPU runtime (T4 minimum, A100 recommended)

**Cost:** Free on Colab (T4), ~$10/mo on Colab Pro (A100)

**Time:** ~2-4 hours for 50 epochs on 10K samples

In [ ]:
# 1. Clone repo + install deps
!git clone https://github.com/aurorav5/rain-blueprint.git
%cd rain-blueprint
!git checkout rain/post-audit-sync
!pip install -q torch onnx onnxruntime onnxscript numpy scipy librosa soundfile

In [ ]:
# 2. Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# 3. Generate training data (heuristic bootstrap)
import json, random, sys, os
sys.path.insert(0, '.')
from ml.rainnet.heuristics import get_heuristic_params, GENRE_PRESETS, PLATFORM_LUFS

GENRES = list(GENRE_PRESETS.keys())
PLATFORMS = list(PLATFORM_LUFS.keys())
N_SAMPLES = 10000

with open('training_manifest.jsonl', 'w') as f:
    for i in range(N_SAMPLES):
        genre = random.choice(GENRES)
        platform = random.choice(PLATFORMS)
        params = get_heuristic_params(genre, platform, vinyl=(platform=='vinyl'))
        # Perturb
        for k in ['mb_ratio_low','mb_ratio_mid','mb_ratio_high']:
            params[k] = max(1.0, min(20.0, params[k] * random.uniform(0.8, 1.2)))
        for j in range(8):
            params['eq_gains'][j] = max(-12, min(12, params['eq_gains'][j] + random.uniform(-2, 2)))
        f.write(json.dumps({'audio_path': f'synthetic/{i:05d}.wav',
                            'genre_label': GENRES.index(genre),
                            'platform_label': PLATFORMS.index(platform),
                            'target_params': params}) + '\n')

print(f'Generated {N_SAMPLES} training samples')

In [ ]:
# 4. Generate synthetic mels (replace with real audio mels for production)
import numpy as np
from pathlib import Path

Path('synthetic').mkdir(exist_ok=True)
with open('training_manifest.jsonl') as f:
    samples = [json.loads(l) for l in f if l.strip()]

for i, s in enumerate(samples):
    mel_path = Path(s['audio_path']).with_suffix('.mel.npy')
    mel_path.parent.mkdir(exist_ok=True)
    np.save(str(mel_path), np.random.randn(128, 128).astype(np.float32) * 0.5)
    if (i+1) % 2000 == 0: print(f'  [{i+1}/{len(samples)}]')

print(f'Done: {len(samples)} mels')

In [ ]:
# 5. Train!
from ml.rainnet.train import train
train('training_manifest.jsonl', output_dir='models/checkpoints', epochs=50, batch_size=64, device='cuda')

In [ ]:
# 6. Export to ONNX
import glob, sys, io
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8', errors='replace')
sys.stderr = io.TextIOWrapper(sys.stderr.buffer, encoding='utf-8', errors='replace')

best = sorted(glob.glob('models/checkpoints/rainnet_v2_epoch_*.pt'))[-1]
print(f'Best checkpoint: {best}')

from ml.rainnet.export import export_onnx
export_onnx(best, 'models/rain_trained.onnx')

In [ ]:
# 7. Verify
import onnxruntime as ort
sess = ort.InferenceSession('models/rain_trained.onnx', providers=['CUDAExecutionProvider','CPUExecutionProvider'])
out = sess.run(None, {
    'mel': np.random.randn(1,1,128,128).astype(np.float32),
    'artist_vec': np.zeros((1,64), dtype=np.float32),
    'genre_id': np.array([3], dtype=np.int64),
    'platform_id': np.array([0], dtype=np.int64),
    'simple_mode': np.ones((1,1), dtype=np.float32),
})
print(f'Output shape: {out[0].shape}')
print(f'target_lufs: {out[0][0][0]:.4f}')
print(f'macros [39:46]: {out[0][0][39:46]}')
print('Verified!')

In [ ]:
# 8. Download the trained model
from google.colab import files
files.download('models/rain_trained.onnx')
print('Download complete. Deploy to production with RAIN_NORMALIZATION_VALIDATED=true')